## USA: How Much Better Off Is a Software Engineer?

SWE mid-career pay as a multiple of each peer role. Values > 1 mean SWE earns more.
Blue-collar and agricultural roles show how far SWE pay diverges from manual work.

In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import plotly.express as px

ROLE_LABELS = {
    'software_engineer': 'Software Engineer', 'lawyer': 'Lawyer',
    'physician': 'Physician', 'financial_analyst': 'Financial Analyst',
    'registered_nurse': 'Registered Nurse', 'civil_engineer': 'Civil Engineer',
    'construction_laborer': 'Construction Laborer', 'farm_worker': 'Farm Worker',
    'manufacturing_worker': 'Manufacturing Worker', 'retail_worker': 'Retail Worker',
}
SECTOR_COLORS = {
    'tech': '#2171b5', 'legal': '#6a51a3', 'healthcare': '#e6550d',
    'finance': '#31a354', 'engineering': '#74c476',
    'blue_collar': '#969696', 'agriculture': '#8c6d31',
    'manufacturing': '#c7e9c0', 'services': '#fdae6b',
}

df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_usa_data.csv')
mid = df[df['career_stage'] == 'mid'].copy()
swe_mid = mid[mid['role'] == 'software_engineer']['median_salary_local'].values[0]
peers = mid[mid['role'] != 'software_engineer'].copy()
peers['swe_multiple'] = (swe_mid / peers['median_salary_local']).round(2)
peers['role_label'] = peers['role'].map(ROLE_LABELS)
peers = peers.sort_values('swe_multiple', ascending=True)

In [2]:
fig = px.bar(
    peers, x='swe_multiple', y='role_label', orientation='h',
    color='sector', color_discrete_map=SECTOR_COLORS,
    title=f'USA: SWE Mid-Career Salary as Multiple of Each Role (BLS OES 2023)<br>'
          f'<sup>SWE median = ${swe_mid:,}/yr — values >1 mean SWE earns more</sup>',
    labels={'swe_multiple': 'SWE / Role salary multiple', 'role_label': 'Role'},
    hover_data={'median_salary_local': ':$,.0f'},
)
fig.add_vline(x=1.0, line_dash='dot', line_color='red', annotation_text='Equal pay')
fig.show()

In [3]:
peers['salary_gap_usd'] = swe_mid - peers['median_salary_local']

fig2 = px.bar(
    peers.sort_values('salary_gap_usd', ascending=True),
    x='salary_gap_usd', y='role_label', orientation='h',
    color='sector', color_discrete_map=SECTOR_COLORS,
    title=f'USA: Annual Salary Gap — SWE Earns More By (USD, 2023)<br>'
          f'<sup>How much more a mid-career SWE makes vs each peer role per year</sup>',
    labels={'salary_gap_usd': 'SWE annual salary advantage (USD)', 'role_label': 'Role'},
)
fig2.show()